
## Workouts BPM Silver Updates
In this lab, you will learn to defines a data processing pipeline for streaming data related to **BPM** (heart rate) measurements and workout sessions and create derived tables like "bpm_silver", "workouts_silver", "workouts_completed", and "workout_bpm".

## Learning Objectives
By the end of this lesson, you should be able to:
- Retrieve rules from a Spark dataset based on a specified topic.
- Process heart rate data by categorizing it.
- Process workout session data, casting timestamps, and ensuring data completeness.
- Determine completed workout sessions by matching "start" and "stop" actions.
- Calculate heart rate data within completed workout sessions, filtering out problematic records.

In [0]:
import dlt
import pyspark.sql.functions as F

## Read rules from Dataset

To read rules from table:
- Define function with name **get_rules** and pass **topic** in parameters.
- Use **`spark.conf.get()`** to retrieves the value of the configuration parameter **`"lookup_db"`**.
- Store the rules in python dictionary in form of key value pair

In [0]:
lookup_db = spark.conf.get("lookup_db")

def get_rules(topic):
    df = spark.read.table(f"{lookup_db}.rules").filter(F.col("topic") == topic)
    rules = {}
    for row in df.collect(): 
        rules[row["name"]] = row["condition"]
    return rules


## Maintaining quality checks in table
Represent the process of processing streaming data for both **heartrate** and **workout-related** data and store it in **"silver"** tables.

Follow these steps to maintain quality check in table:
- Read a stream named **"valid_bpm"** select columns: "device_id", "time", "heartrate".
- Add a new column **"bpm_check"** based on the **"heartrate"** value and add watermark with 30 sec
- Drop duplicate records based on "device_id" and "time
- Same for **"valid_workouts"** select columns: "user_id", "workout_id", "timestamp" (casted as "time"), "action", "session_id" apply watermarking on time column and drop duplicates
- Create a SQL query that performs a left join between **"workouts_silver"** for "start" actions and **"workouts_silver"** for "stop" actions 
- Select columns from "a" and "b", calculating the "in_progress" status.
- Similarly, create a SQL query that joins **"bpm_silver"** with **"workouts_completed"** and **"user_lookup"**.
- Select appropriate columns from the joined data.
- Apply a filter to select only records where **"bpm_check"** is 'OK

In [0]:
@dlt.table(table_properties={"quality": "silver"})
def bpm_silver():
    return (
        dlt.read_stream("valid_bpm")
          .select("device_id", "time", "heartrate")
          .withColumn("bpm_check", F.when(F.col("heartrate") <= 0, "Negative BPM").otherwise("OK"))
          .withWatermark("time", "30 seconds")
          .dropDuplicates(["device_id", "time"])
    )


@dlt.table(table_properties={"quality": "silver"})
def workouts_silver():
    return (
        dlt.read_stream("valid_workouts")
          .select("user_id", "workout_id", 
                  F.col("timestamp").cast("timestamp").alias("time"), 
                  "action", "session_id")
          .withWatermark("time", "30 seconds")
          .dropDuplicates(["user_id", "time"])
    )


@dlt.table
def workouts_completed():
    return spark.sql(f"""
      SELECT a.user_id, a.workout_id, a.session_id, a.start_time start_time, b.end_time end_time, a.in_progress AND (b.in_progress IS NULL) in_progress
      FROM (
        SELECT user_id, workout_id, session_id, time start_time, null end_time, true in_progress
        FROM LIVE.workouts_silver
        WHERE action = "start") a
      LEFT JOIN (
        SELECT user_id, workout_id, session_id, null start_time, time end_time, false in_progress
        FROM LIVE.workouts_silver
        WHERE action = "stop") b
      ON a.user_id = b.user_id AND a.session_id = b.session_id
    """)


@dlt.table
def workout_bpm():
    return spark.sql(f"""
      SELECT d.user_id, d.workout_id, d.session_id, time, heartrate
      FROM STREAM(LIVE.bpm_silver) c
      INNER JOIN (
        SELECT a.user_id, b.device_id, workout_id, session_id, start_time, end_time
        FROM LIVE.workouts_completed a
        INNER JOIN LIVE.user_lookup b
        ON a.user_id = b.user_id) d
      ON c.device_id = d.device_id AND time BETWEEN start_time AND end_time
      WHERE c.bpm_check = 'OK'
    """)